Reissner Mindlin Plate : Modal Analysis with Fenicsx 
=================

First of all, this work is base on Jeremie Bleyer's work which can be found with this link : [Fenicsx Tour : Reissner-Mindlin plates ](https://bleyerj.github.io/comet-fenicsx/intro/plates/plates.html)

Governor Equation
-----------------

Les Equations BLALBLALBLALa


Implementation Start 
-----------------
Importation des bibliothéque utile

In [1]:
import numpy as np
import ufl
import basix

from mpi4py import MPI
from dolfinx import fem, io
import dolfinx.fem.petsc
from dolfinx.fem import Constant, dirichletbc, Function, functionspace, locate_dofs_topological
import dolfinx.mesh
from dolfinx.io import XDMFFile

Implentation des paramétres de la plaque étudié

In [2]:
L  = 0.3 # Longueur de la plaque
W_dim = 0.24 # Largeur de la plaque
N = 20 # Nombre de subdivision de la plaque
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD, [[0, 0], [L, W_dim]], [N, N] , cell_type=dolfinx.mesh.CellType.triangle, diagonal=dolfinx.mesh.DiagonalType.crossed
) # Création du "mesh" de la plaque elle nous servira tous le long du programme pour appeler notre forme


# material parameters
thick = 0.00105 # épaisseur de la plaque
E = 210.0e3 # Module de Young du matériaus [MPa]
nu = 0.3 # Coef de poisson du matériau
rho = 7.7777 # Masse volumique [kg/L]

## Création des des matrices et des constantes :
- Matrice des Moments
$$
\mathbf{D} = D \begin{bmatrix}
1 & \nu & 0 \\
\nu & 1 & 0 \\
0 & 0 & \frac{1-\nu}{2}
\end{bmatrix} 
\quad \text{avec} \quad 
D = \frac{E \times h^3}{12(1-\nu^2)}
$$
- Constante des efforts de cisaillement
$$
F = \frac{5}{6} \times \frac{E \times h}{2(1 + \nu)}
$$
- Matrice de Masse
$$
\mathbf{M} = M \begin{bmatrix}
1 & 0 & 0 \\
0 & \frac{h^2}{12} & 0 \\
0 & 0 & \frac{h^2}{12}
\end{bmatrix}
\quad \text{avec} \quad
M = \rho h
$$

In [5]:
# bending stiffness
D = fem.Constant(domain, E * thick**3 / (1 - nu**2) / 12.0)
# shear stiffness
F = fem.Constant(domain, E / 2 / (1 + nu) * thick * 5.0 / 6.0)
# Inertilal constant
M = fem.Constant(domain , rho * thick)

def curvature(u):
    (w, theta) = ufl.split(u)
    return ufl.as_vector(
        [theta[0].dx(0), theta[1].dx(1), theta[0].dx(1) + theta[1].dx(0)]
    )


def shear_strain(u):
    (w, theta) = ufl.split(u)
    return ufl.grad(w) - theta


def bending_moment(u):
    DD = ufl.as_matrix([[D, nu * D, 0], [nu * D, D, 0], [0, 0, D * (1 - nu) / 2.0]])
    return ufl.dot(DD, curvature(u))

def inertial(u) :
    MM = ufl.as_matrix([[M, 0, 0], [0, M*thick**2/12, 0], [0, 0, M*thick**2/12]])
    return ufl.dot(MM, u)


def shear_force(u):
    return F * shear_strain(u)


On crée un vecteur avec 1 déplacement vertical et 2 rotation autour de la courbure :

$U_e = U_3$

$T_e = \begin{bmatrix} \phi_1  \\ \phi_2 \end{bmatrix} $

In [ ]:

Ue = basix.ufl.element("P", domain.basix_cell(), 2)
Te = basix.ufl.element("P", domain.basix_cell(), 1, shape=(2,))
V = fem.functionspace(domain, basix.ufl.mixed_element([Ue, Te]))